# LC 76 — Minimum Window Substring
**Difficulty:** Hard &nbsp;|&nbsp; **Category:** Sliding Window
**Pattern:** Expand Right to Find Valid, Shrink Left to Minimise

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Expand the right pointer
until the window contains all required characters.
Then shrink from the left as far as possible while
the window stays valid. Record the smallest valid
window found.
</div>

## Official Problem Statement

Given two strings `s` and `t` of lengths `m` and
`n` respectively, return the minimum window
substring of `s` such that every character in `t`
(including duplicates) is included in the window.
If there is no such substring, return the empty
string `""`.

**Example 1:**
```
Input:  s = "ADOBECODEBANC", t = "ABC"
Output: "BANC"
```
**Example 2:**
```
Input:  s = "a", t = "a"
Output: "a"
```
**Example 3:**
```
Input:  s = "a", t = "aa"
Output: ""
```

**Constraints:**
- `m == s.length`,  `n == t.length`
- `1 <= m, n <= 10^5`
- `s` and `t` consist of uppercase and lowercase
  English letters
- The test cases are generated so that the answer is
  unique

## What This Is Actually Asking

Find the shortest segment of s that contains every
character from t (with the right counts — if t has
two 'A's, the window must have at least two 'A's).
The segment must be contiguous.
Return the actual substring, not just its length.

## Walk Through an Example by Hand

```
s = "ADOBECODEBANC"   t = "ABC"
need = {A:1, B:1, C:1}   required=3  have=0
left=0  result=""

Expand right:
r=0 'A'  window={A:1}  A meets need -> have=1
r=1 'D'  window={A:1,D:1}  not in need
r=2 'O'  not in need
r=3 'B'  window={..B:1}  B meets need -> have=2
r=4 'E'  not in need
r=5 'C'  window={..C:1}  C meets need -> have=3

have==required -> VALID window: s[0..5]="ADOBEC" len=6
result="ADOBEC"

Shrink left:
  left=0 'A': window[A]=1==need[A] -> removing breaks it
  have=2 -> stop shrinking  left=1

Expand right:
r=6 'O'  r=7 'D'  r=8 'E'  r=9 'B'  window[B]=2
r=10 'A'  window[A]=1 (counted from left=1)
  Actually A was at index 0 which is outside window now.
  A meets need again -> have=3

VALID window: s[1..10]="DOBECODEBA" -> shrink
... eventually shrinks to s[9..12]="BANC" len=4

result="BANC"
```

## The Picture

```
s = A D O B E C O D E B A N C
    0 1 2 3 4 5 6 7 8 9 ...12

t = "ABC"   need = {A:1, B:1, C:1}

Phase 1 — EXPAND right until window is valid:
  [A D O B E C]  contains A,B,C -> valid!
   L           R

Phase 2 — SHRINK left while still valid:
  [A D O B E C]  remove A -> A count drops to 0 -> invalid
   L is stuck    record "ADOBEC" len=6

Continue expanding right...eventually:
  [...B A N C]   contains A,B,C -> valid!
         L    R
  shrink all the way:
  [B A N C]      still valid  len=4  <- new best
   L       R

Two counters track validity:
  have     = how many distinct t-chars are satisfied
  required = len(need)  (unique chars needed)
  valid when have == required
```

## When To Use This Pattern

- When you see **minimum window containing all of t**,
  think **expand right to find valid, shrink left
  to minimise**
- When validity depends on character counts (not
  just presence), think **have/required counter pair**
- When a character's count in the window meets the
  need, think **increment have** (only on the exact
  threshold — not every time count grows)
- When removing a character drops its count below
  need, think **decrement have and stop shrinking**

## The Approach

Build a need map from t and set required to the
number of unique characters needed.
Expand right: add each character to the window map;
when its count first meets the need, increment have.
When have equals required, the window is valid —
record it if it is the smallest so far, then shrink
from the left until the window becomes invalid again.
Repeat until right reaches the end of s.

In [1]:
from collections import Counter  # build need map from t

In [13]:
def test_harness(func):
    tests = [
        # (s, t, expected)
        ("ADOBECODEBANC", "ABC",  "BANC"),
        ("a",             "a",    "a"),
        ("a",             "aa",   ""),    # t longer than s
        ("aa",            "aa",   "aa"),  # exact match
        ("bba",           "ab",   "ba"),
        ("abc",           "b",    "b"),   # single char t
        ("cabwefgewcwaefgcf", "cae", "cwae"),
        ("OUZODYXAZV",    "XZ",   "XAZ"),
    ]

    passed = 0
    for i, (s, t, expected) in enumerate(tests):
        result = func(s, t)
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"s={s!r} t={t!r} | "
            f"expected={expected!r} | got={result!r}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [16]:
import math
def minWindow(s: str, t: str) -> str:
    """
    Return the minimum window in s containing all of t.

    Build need map from t; set required = len(need).
    Expand right: when window count meets need for a
    char, increment have. While have==required, record
    smallest window then shrink left (decrement have
    when a char drops below need). Return best window.

    Time:  O(n + m) — each pointer moves at most n
    Space: O(n + m) — window and need maps
    """
    tCounts = Counter (t)
    sCounts = {}
    need, have, l , minWinl, minr, minl = len(tCounts), 0 , 0, math.inf, -1,-1
    for r, c in enumerate(s):
        sCounts[c] = sCounts.get(c,0) + 1
        if c in tCounts and tCounts[c] == sCounts[c]:
            have += 1
        while have == need:
            wl = r -l +1
            if wl < minWinl:
                minWinl = wl
                minr = r
                minl = l
            ch = s[l]
            sCounts[ch] = sCounts.get(ch,0) - 1
            if ch in tCounts and sCounts[ch] < tCounts[ch]:
                have -= 1
            l += 1
    return s[minl:minr+1] if minr != -1 else ""
            
r"""
BANC
a

ba
Test 1: PASSED | s='ADOBECODEBANC' t='ABC' | expected='BANC' | got='BANC'
Test 2: PASSED | s='a' t='a' | expected='a' | got='a'
Test 3: PASSED | s='a' t='aa' | expected='' | got=''
Test 4: PASSED | s='aa' t='aa' | expected='aa' | got='aa'
Test 5: PASSED | s='bba' t='ab' | expected='ba' | got='ba'
Test 6: PASSED | s='abc' t='b' | expected='b' | got='b'
Test 7: PASSED | s='cabwefgewcwaefgcf' t='cae' | expected='cwae' | got='cwae'
Test 8: PASSED | s='OUZODYXAZV' t='XZ' | expected='XAZ' | got='XAZ'

8/8 tests passed
"""

# Quick debug — run this cell while building
print(minWindow("ADOBECODEBANC", "ABC"))  # "BANC"
print(minWindow("a", "a"))                # "a"
print(minWindow("a", "aa"))               # ""
print(minWindow("bba", "ab"))             # "ba"
test_harness(minWindow)

BANC
a

ba
Test 1: PASSED | s='ADOBECODEBANC' t='ABC' | expected='BANC' | got='BANC'
Test 2: PASSED | s='a' t='a' | expected='a' | got='a'
Test 3: PASSED | s='a' t='aa' | expected='' | got=''
Test 4: PASSED | s='aa' t='aa' | expected='aa' | got='aa'
Test 5: PASSED | s='bba' t='ab' | expected='ba' | got='ba'
Test 6: PASSED | s='abc' t='b' | expected='b' | got='b'
Test 7: PASSED | s='cabwefgewcwaefgcf' t='cae' | expected='cwae' | got='cwae'
Test 8: PASSED | s='OUZODYXAZV' t='XZ' | expected='XAZ' | got='XAZ'

8/8 tests passed


In [ ]:
# Uncomment and run when solution is ready
# test_harness(minWindow)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force — all substrings | O(n² × m) | O(m) |
| Sliding window with have/required | O(n + m) | O(n + m) |

Each character in s is visited at most twice (once
by right, once by left) — the inner while loop is
amortized O(1) per character.

## Real World Connection

At Citi, log correlation searches for the shortest
time window in a server event log that contains all
required diagnostic events — authentication, query
execution, and result delivery — to reconstruct a
failed transaction.
The minimum window substring maps directly: s is
the event log, t is the set of required event types,
and the answer is the tightest time bracket that
captures the full lifecycle.
In the ETL pipeline, the same pattern finds the
shortest segment of a Kafka consumer batch that
covers all required partition offsets before
committing the checkpoint.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra